In [ ]:
import math
# import numpy as np


# Common

In [ ]:
class Vector:
    X: float
    Y: float
    
    def __init__(self, x, y):
        self.X = x
        self.Y = y
    
    def Norm(self):
        x = self.X
        y = self.Y
        return math.sqrt(x*x + y*y)
    
    # 實現相減運算 (self - other)
    def __sub__(self, other):
        if isinstance(other, Vector):
            return Vector(self.X - other.X, self.Y - other.Y)
        return NotImplemented
    

In [ ]:
def Dist(p1, p2):
    dx = p1[0] - p2[0]
    dy = p1[1] - p2[1]
    return math.sqrt(dx*dx + dy*dy)

# CALIBRATION

In [ ]:
class Calib:
    Rows = 22
    Cols = 7
    Pitch = Vector(9.6, 10.5)

    S1 = {
        'camera' : [
            Vector( 5051.5,  3956.8),
            Vector(13872.4,  4025.8),
            Vector(13932.7, 33479.8),
            Vector( 5063.5, 33472.0)
        ],
        'motor' : [
            Vector(-1016.960, -213.238),
            Vector( -959.270, -213.238),
            Vector( -959.540, 6.835),
            Vector(-1017.018, 6.835)
        ]
    }

    S2 = {
        'camera' : [
            Vector( 5045.3,  3934.1),
            Vector(13916.4,  4005.7),
            Vector(13953.9, 34632.2),
            Vector( 5092.5, 34567.8)
        ],
        'motor' : [
            Vector(-1015.316, -284.640),
            Vector( -957.385, -284.640),
            Vector( -957.215, -63.801),
            Vector(-1014.877, -63.801)
        ]
    }

    def Width(self):
        return (self.Cols-1) * self.Pitch.X
    
    def Height(self):
        return (self.Rows-1) * self.Pitch.Y
    
    def Corners(self, sName:str, coord:str):
        S = self.S1 if sName=="S1" else self.S2
        corners = S[coord]
        return corners


In [ ]:
calibBoard = Calib()
W = calibBoard.Width()
H = calibBoard.Height()

print(f"理想 W= {W:0.3f}")
print(f"理想 H= {H:0.3f}")
print('')

for sName in ['S1', 'S2']:
    corners = calibBoard.Corners('S1','motor')
    w1 = (corners[0]-corners[1]).Norm()
    w2 = (corners[3]-corners[2]).Norm()
    h1 = (corners[0]-corners[3]).Norm()
    h2 = (corners[1]-corners[2]).Norm()
    print(f"{sName} 馬達 ΔW: {w1-W:0.3f}, {w2-W:0.3f}")
    print(f"{sName} 馬達 ΔH: {h1-H:0.3f}, {h2-H:0.3f}")
    print('')

    corners = calibBoard.Corners(sName,'camera')
    w1 = (corners[0]-corners[1]).Norm()
    w2 = (corners[3]-corners[2]).Norm()
    h1 = (corners[0]-corners[3]).Norm()
    h2 = (corners[1]-corners[2]).Norm()
    print(f"{sName} 相機 W: {w1:0.0f}, {w2:0.0f}")
    print(f"{sName} 相機 H: {h1:0.0f}, {h2:0.0f}")
    print('')

In [ ]:
S1 = calibBoard.Corners('S1','motor')
S2 = calibBoard.Corners('S2','motor')

for c in range(4):
    dx = S1[c].X - S2[c].X
    dy = S1[c].Y - S2[c].Y
    print(f"[c={c}] 馬達S1 馬達S2 : (ΔX,ΔY) = ({dx:0.3f}, {dy:0.3f})")


# Migration

In [ ]:
class Migration(Calib):
    Rows = 16
    Cols = 6
    Pitch = Vector(8.85, 9.85)

    S1_REF = { "camera": Vector(5693.4, 7739.6), "motor": Vector(-1012.794, -185.769) }
    S2_REF = { "camera": Vector(5693.4, 7739.6), "motor": Vector(-1011.073, -257.154) }
    
    def REF(self, sName, coord):
        ref = self.S1_REF if sName=='S1' else self.S2_REF
        return ref[coord]

In [ ]:
mig = Migration()

In [ ]:
s1 = mig.REF('S1','motor')
s2 = mig.REF('S2','motor')
dx = s1.X - s2.X
dy = s1.Y - s2.Y
print(f"[REF] 馬達S1 馬達S2 : (ΔX,ΔY) = ({dx:0.3f}, {dy:0.3f})")

In [ ]:
print(f"理想 W= {mig.Width():0.3f}")
print(f"理想 H= {mig.Height:0.3f}")
print('')